# Earthquake Risk Model
## File: models/train_earthquake.ipynb

**Run karo:** Kernel → Restart & Run All
**Output:** `earthquake_model.pkl` → Copy karo `backend/` folder mein

**India seismic zones:**
- Zone V (highest): NE India, J&K, Himachal
- Zone IV: Indo-Gangetic plains, Delhi
- Zone III: Rest of India

## Step 1 — Libraries

In [1]:
import numpy as np
import pandas as pd
import pickle
import warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, f1_score, roc_auc_score)
print("Ready!")

Ready!


## Step 2 — Dataset + Train + Save

In [2]:
def generate_earthquake_data(n=5000, seed=45):
    
    #Earthquake risk dataset for Indian subcontinent.
    #Key factors: proximity to fault, tectonic stress, historical frequency.
    
    #Real sources:
    #- USGS Earthquake Catalog: https://earthquake.usgs.gov
    #- IMD Seismology: https://seismo.gov.in
    #- GSI Fault Maps: https://www.gsi.gov.in
    
    np.random.seed(seed)
    
    fault_dist    = np.random.exponential(50, n).clip(0.5, 500)
    seismic_30d   = np.random.uniform(0, 50, n)
    tectonic      = np.random.uniform(0, 100, n)
    depth_km      = np.random.uniform(5, 300, n)
    mag_recent    = np.random.uniform(0, 6, n)
    ground_vel    = np.random.uniform(0, 10, n)
    radon         = np.random.uniform(0, 100, n)
    soil_code     = np.random.uniform(1, 4, n)
    lat           = np.random.uniform(8, 35, n)
    lon           = np.random.uniform(68, 97, n)
    hist_freq     = np.random.uniform(0, 10, n)
    
    p  = 0.03
    p += 0.30 * (1 - fault_dist / 500).clip(0, 1)
    p += 0.20 * (tectonic / 100)
    p += 0.15 * (seismic_30d / 50)
    p += 0.12 * (mag_recent / 6)
    p += 0.10 * (radon / 100)
    p += 0.08 * (soil_code / 4)
    p += 0.05 * (hist_freq / 10)
    p += np.random.normal(0, 0.04, n)
    p  = p.clip(0, 1)
    earthquake = (p > np.percentile(p, 84)).astype(int)
    
    df = pd.DataFrame({
        'fault_distance_km':fault_dist.round(2),
        'seismic_activity_30d':seismic_30d.round(1),
        'tectonic_stress':tectonic.round(2),
        'depth_km':depth_km.round(1),
        'magnitude_recent':mag_recent.round(2),
        'ground_velocity':ground_vel.round(3),
        'radon_emission':radon.round(2),
        'soil_type_code':soil_code.round(1),
        'latitude':lat.round(4),
        'longitude':lon.round(4),
        'historical_freq':hist_freq.round(2),
        'earthquake':earthquake
    })
    return df

FEATURES = [
    'fault_distance_km','seismic_activity_30d','tectonic_stress','depth_km',
    'magnitude_recent','ground_velocity','radon_emission','soil_type_code',
    'latitude','longitude','historical_freq',
    'proximity_stress','seismic_energy','amplification_risk'
]

def engineer(df):
    df = df.copy()
    df['proximity_stress']   = df['tectonic_stress'] / (df['fault_distance_km'] + 1)
    df['seismic_energy']     = df['magnitude_recent'] * df['seismic_activity_30d']
    df['amplification_risk'] = df['soil_type_code'] * (1 - df['depth_km'] / 300)
    return df

df     = generate_earthquake_data()
df_eng = engineer(df)
X      = df_eng[FEATURES]; y = df_eng['earthquake']
X_tr,X_te,y_tr,y_te = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

model = RandomForestClassifier(n_estimators=200,max_depth=15,
    class_weight='balanced',random_state=42,n_jobs=-1)
model.fit(X_tr, y_tr)
pred  = model.predict(X_te)
proba = model.predict_proba(X_te)[:,1]

print("EARTHQUAKE MODEL RESULTS")
print(f"F1-Score : {f1_score(y_te, pred):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_te, proba):.4f}")
print()
print(classification_report(y_te, pred, target_names=['No EQ','Earthquake']))

with open('earthquake_model.pkl','wb') as f:
    pickle.dump({'model':model,'features':FEATURES,'version':'1.0','disaster':'earthquake'},f)
print("earthquake_model.pkl saved! Copy to backend/ folder.")

EARTHQUAKE MODEL RESULTS
F1-Score : 0.6618
ROC-AUC  : 0.9603

              precision    recall  f1-score   support

       No EQ       0.92      0.97      0.95       840
  Earthquake       0.79      0.57      0.66       160

    accuracy                           0.91      1000
   macro avg       0.86      0.77      0.80      1000
weighted avg       0.90      0.91      0.90      1000

earthquake_model.pkl saved! Copy to backend/ folder.
